# 02 — Distribuciones y balance
**Autor:** Giuliano Crenna, Juan Ignacio Pace (UGR)
**Fecha:** 2026-09-03
**Descripción:** Barplots de clases, heatmap fuente × clase, comparación de
longitudes entre clases.
## Parámetros
- `DATA_DIR`, `SEED`, `OUT_DIR` (papermill).


In [0]:
# %% [code]
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
DATA_DIR = Path(os.environ.get("DATA_DIR", "./data"))
SEED = int(os.environ.get("SEED", 42))
OUT_DIR = Path(os.environ.get("OUT_DIR", "reports"))
np.random.seed(SEED)
df = pd.read_parquet(DATA_DIR / "processed" / "corpus_v1.parquet")
print(f"corpus: {len(df)} filas")


In [0]:
# %% [code]
# Distribución de clases (global).
fig, ax = plt.subplots(figsize=(7, 4))
counts = df["label"].value_counts().sort_index()
ax.bar(counts.index.astype(str), counts.values)
ax.set_xlabel("label (0=control, 1=moderado, 2=depresivo)")
ax.set_ylabel("# documentos")
ax.set_title("Distribución global de clases")
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.tight_layout()
out = OUT_DIR / "figures" / "eda_02_distribucion_clases.png"
out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=120)
plt.show()


In [0]:
# %% [code]
# Heatmap fuente × label.
pivot = df.groupby(["source", "label"]).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(pivot, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("Heatmap fuente × label")
plt.tight_layout()
out = OUT_DIR / "figures" / "eda_02_heatmap_fuente_label.png"
plt.savefig(out, dpi=120)
plt.show()


In [0]:
# %% [code]
# Longitudes comparadas por clase.
df["len_tokens"] = df["text_clean"].fillna("").str.split().str.len()
fig, ax = plt.subplots(figsize=(8, 4))
for lab, sub in df.groupby("label"):
    ax.hist(sub["len_tokens"].clip(upper=150), bins=40, alpha=0.5, label=f"label={lab}")
ax.set_xlabel("# tokens (clip a 150)")
ax.set_ylabel("frecuencia")
ax.set_title("Distribución de longitudes por clase")
ax.legend()
plt.tight_layout()
out = OUT_DIR / "figures" / "eda_02_longitudes_por_clase.png"
plt.savefig(out, dpi=120)
plt.show()


## Conclusiones
- Documentar balance: si una clase tiene <10% de representación, vamos a
  necesitar class weights en la etapa 4.
- Si el heatmap muestra que una fuente aporta casi todos los ejemplos de
  una clase, hay riesgo de leakage del estilo de escritura — considerar
  drop o re-muestreo.
